# KORMo 1B — 벤치마크 평가 (Colab)

`01_kormo_1B_pretrain_colab_a100.ipynb`로 학습해 Drive에 저장한 모델(`output/final/`)을
[lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)로 평가합니다 —
HF Open LLM Leaderboard·Open Ko-LLM Leaderboard가 쓰던 것과 같은 도구입니다.

**전제**: Drive `MyDrive/kormo-1B-PT/output/final/`에 학습된 모델 존재 (1번 노트북 완주)
— 구버전 레이아웃(`kormo-1B-PT/final/`)도 자동 인식합니다

**런타임**: L4 GPU면 충분합니다 (A100 불필요 — 평가는 forward만 하므로)

구성:
1. **Held-out perplexity** — 학습에 쓰지 않은 한국어 위키 텍스트에 대한 PPL. 튜토리얼 스케일에서 가장 민감한 지표
2. **KoBEST + HAE-RAE** — loglikelihood 기반이라 작은 모델도 측정 가능 (~10분)
3. **KMMLU** (선택) — 45과목 전문지식. 오래 걸리고 이 스케일에선 랜덤 수준이 예상되므로 기본 off

In [ ]:
!nvidia-smi -L

## 0. 환경 준비 + 모델 로드

평가는 문서 단위 forward라 패킹 마스크가 필요 없으므로 `flex_attention` 대신 **sdpa**로 로드합니다 (더 빠르고 컴파일 오버헤드 없음).

In [ ]:
import os
# HF 다운로드 경로 — 토큰 유무에 따라 분기 (huggingface_hub import 전에 결정해야 함):
# - HF_TOKEN 있음: Xet 네이티브 프로토콜 사용. 브리지 CDN(us.gcp.cdn.hf.co)을 거치지 않아
#   Colab(GCP 리전)에서 발생하는 403 SignatureError(invalid key pair id) 장애를 우회
# - HF_TOKEN 없음: Xet 비활성 → 일반 HTTP 폴백 (익명 Xet 접근은 cas-server 401이 나므로)
FORCE_DISABLE_XET = False   # 토큰이 정상인데도 다운로드가 계속 실패하면 True로 바꿔 일반 HTTP로 강제 폴백

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("HF_TOKEN 로드됨 — Xet 네이티브 경로 사용")
except Exception:
    os.environ['HF_HUB_DISABLE_XET'] = '1'
    print("HF_TOKEN 없음 — Xet 비활성 (브리지 CDN 폴백)")

if FORCE_DISABLE_XET:
    os.environ['HF_HUB_DISABLE_XET'] = '1'
    print("FORCE_DISABLE_XET — Xet 비활성 (일반 HTTP 폴백)")

!git clone -q https://github.com/MLP-Lab/KORMo-tutorial.git /content/KORMo-tutorial
%pip install -q "transformers==4.57.1" "datasets>=4.1.1" lm-eval hf_xet

import sys, json
sys.path.append('/content/KORMo-tutorial/src')

In [ ]:
import torch
from google.colab import drive
from transformers import AutoTokenizer
from kormo.model._modeling_kormo import KORMoForCausalLM

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/kormo-1B-PT'
FINAL_DIR = os.path.join(BASE_DIR, 'output', 'final')
if not os.path.isdir(FINAL_DIR) and os.path.isdir(os.path.join(BASE_DIR, 'final')):
    FINAL_DIR = os.path.join(BASE_DIR, 'final')   # 구버전 레이아웃 (1번 노트북 재실행 시 output/으로 이동됨)
assert os.path.isdir(FINAL_DIR), (
    f"{FINAL_DIR} 가 없습니다 — 1번 노트북(사전학습)을 먼저 완주해서 final/을 만들어야 합니다."
)
print("모델 경로:", FINAL_DIR)

model = KORMoForCausalLM.from_pretrained(
    FINAL_DIR, dtype=torch.bfloat16, _attn_implementation='sdpa',
).to('cuda').eval()

try:
    tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained('KORMo-Team/KORMo-tokenizer')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"로드 완료: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params")

## 1. Held-out perplexity (한국어 위키)

학습 데이터(KORMo-tutorial-datasets)에 없는 한국어 위키백과 문서 200개에 대해 PPL을 계산합니다.

해석 기준:
- 무작위 초기화 모델: PPL ≈ vocab 크기(125,184) 수준
- 학습이 조금이라도 됐다면: 수백~수천 대로 하락
- 추가 학습(continue)을 반복하며 **이 숫자가 계속 내려가는지**가 튜토리얼 스케일에서 가장 좋은 진행 지표입니다

In [ ]:
import math, time
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

# 스트리밍(HTTP range 요청)은 Xet 브리지 CDN(us.gcp.cdn.hf.co)을 타는데, Colab(GCP 리전)에서
# 403 SignatureError가 나므로 스트리밍 대신 샤드 1개(400MB)를 hf_hub_download로 통째로 받아
# 로컬에서 읽음 — 통짜 다운로드는 Xet 네이티브 경로라 브리지 CDN을 우회함.
# 문서 선정(앞 500개 → 500자 초과 필터 → 200개)은 기존 스트리밍 방식과 동일 → 평가 이력 비교 가능
def retry_wait(e, attempt, max_attempts=5):
    """에러 내용을 그대로 출력하고 상태 코드에 맞게 대기/중단 결정."""
    code = getattr(getattr(e, 'response', None), 'status_code', None)
    print(f"[{attempt}/{max_attempts}] 실패: {type(e).__name__}(HTTP {code}) — {str(e)[:300]}")
    if code == 401:
        raise RuntimeError("401 인증 실패 — Colab 보안 비밀 HF_TOKEN이 만료/무효입니다. 재발급 후 다시 실행하세요") from e
    wait = 90 * attempt if code == 429 else 30   # 429(rate limit)는 길게 기다려야 풀림
    print(f"  → {wait}초 후 재시도")
    time.sleep(wait)

texts = None
for attempt in range(1, 6):
    try:
        shard = hf_hub_download('wikimedia/wikipedia',
                                '20231101.ko/train-00000-of-00003.parquet',
                                repo_type='dataset')
        rows = []
        for batch in pq.ParquetFile(shard).iter_batches(batch_size=500, columns=['text']):
            rows += batch['text'].to_pylist()
            if len(rows) >= 500:
                break
        texts = [t for t in rows[:500] if len(t) > 500][:200]
        break
    except Exception as e:
        retry_wait(e, attempt)
assert texts, ("5회 재시도 실패 — 위 에러 메시지 확인: 403이면 환경 셀의 FORCE_DISABLE_XET=True로 재시도, "
               "429면 시간을 두고 재시도, 5xx면 status.huggingface.co 확인")
print(f"평가 문서: {len(texts)}개")

total_nll, total_tokens = 0.0, 0
with torch.no_grad():
    for text in texts:
        ids = tokenizer.encode(text, return_tensors='pt')[:, :2048].to('cuda')
        if ids.shape[1] < 16:
            continue
        loss = model(input_ids=ids, labels=ids).loss  # 토큰 평균 NLL
        n = ids.shape[1] - 1
        total_nll += loss.item() * n
        total_tokens += n

ppl = math.exp(total_nll / total_tokens)
print(f"\nHeld-out PPL (한국어 위키, {total_tokens:,} 토큰): {ppl:,.1f}")

## 2. KoBEST + HAE-RAE (lm-evaluation-harness)

객관식 벤치마크는 생성이 아니라 **선택지별 loglikelihood 비교**로 채점되므로, 옹알이 단계의 base 모델도 평가할 수 있습니다.

무작위 기준선 (이 값 근처면 "아직 못 푸는 것", 유의미하게 위면 학습 신호):

| 태스크 | 방식 | 랜덤 기준선 |
|---|---|---|
| kobest_boolq / copa / wic / sentineg | 2지선다 | 0.50 |
| kobest_hellaswag | 4지선다 | 0.25 |
| haerae (5개 하위 태스크) | 5지선다 | ~0.20 |

In [ ]:
import time
import lm_eval
from lm_eval.models.huggingface import HFLM

lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=16)

# HF CDN 순단(IncompleteRead 등)에 대비한 자동 재시도 —
# datasets가 받은 파일을 캐시하므로 재시도는 실패 지점부터 이어짐
results = None
for attempt in range(1, 6):
    try:
        results = lm_eval.simple_evaluate(
            model=lm,
            tasks=['kobest', 'haerae'],   # 그룹명 — kobest 5종 + haerae 5종 전체
            num_fewshot=0,
        )
        break
    except Exception as e:
        retry_wait(e, attempt)   # 1번 섹션 셀에서 정의 — 에러 출력 + 상태 코드별 대기
assert results is not None, "5회 재시도 실패 — 위 에러 메시지를 확인하세요"

print(lm_eval.utils.make_table(results))

## 3. (선택) KMMLU — 45과목 전문지식

`RUN_KMMLU = True`로 바꾸면 실행됩니다. 35k 문항 × 4지선다라 L4에서 1시간 이상 걸리고,
튜토리얼 스케일 모델은 0.25(랜덤) 부근이 예상되므로 기본은 꺼져 있습니다.
KORMo-10B 논문 점수와 비교하려면 논문과 같은 5-shot 설정을 쓰세요.

In [ ]:
RUN_KMMLU = False

if RUN_KMMLU:
    kmmlu_results = lm_eval.simple_evaluate(
        model=lm,
        tasks=['kmmlu'],
        num_fewshot=5,   # KORMo 논문 설정
    )
    print(lm_eval.utils.make_table(kmmlu_results))

## 4. 결과 저장

결과를 Drive에 JSON으로 남깁니다. continue 모드로 추가 학습을 반복할 때
이 파일들을 비교하면 학습량 대비 성능 곡선을 그릴 수 있습니다.

In [ ]:
from datetime import datetime

EVAL_DIR = '/content/drive/MyDrive/kormo-1B-PT/evals'
os.makedirs(EVAL_DIR, exist_ok=True)

stamp = datetime.now().strftime('%Y%m%d_%H%M')
summary = {
    'timestamp': stamp,
    'heldout_ppl_ko_wiki': ppl,
    'benchmarks': {k: v for k, v in results['results'].items()},
}
if RUN_KMMLU:
    summary['kmmlu'] = {k: v for k, v in kmmlu_results['results'].items()}

out_path = f'{EVAL_DIR}/eval_{stamp}.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)
print("저장:", out_path)

## 5. 결과 시각화

`evals/` 폴더의 모든 `eval_*.json`을 읽어 그립니다:
1. **최신 평가** — 태스크별 정확도 vs 무작위 기준선 (기준선 위로 얼마나 떴는지가 학습 신호)
2. **평가 이력 추이** — 추가 학습(continue)을 반복할수록 PPL이 내려가고 정확도가 올라가는지

평가를 다시 돌리지 않고 이 섹션만 실행해도 됩니다 (위에서 Drive 마운트와 `EVAL_DIR` 셀까지만 실행돼 있으면 됨).

In [ ]:
import glob as _glob
import matplotlib.pyplot as plt

# ---- 누적된 평가 JSON 로드 ----
def _acc_of(task_dict):
    # lm-eval 0.4.x: {'acc,none': 0.51, 'acc_stderr,none': ..., 'alias': ...}
    for k, v in task_dict.items():
        if k.split(',')[0] in ('acc', 'acc_norm') and 'stderr' not in k and isinstance(v, (int, float)):
            return float(v)
    return None

runs = []
for path in sorted(_glob.glob(f'{EVAL_DIR}/eval_*.json')):
    with open(path, encoding='utf-8') as f:
        d = json.load(f)
    accs = {t: _acc_of(v) for t, v in d.get('benchmarks', {}).items()}
    runs.append({
        'stamp': d['timestamp'],
        'ppl': d.get('heldout_ppl_ko_wiki'),
        'accs': {t: a for t, a in accs.items() if a is not None},
    })

assert runs, f"{EVAL_DIR}에 eval_*.json이 없습니다 — 4번 섹션(결과 저장)까지 먼저 실행하세요"
print(f"평가 기록 {len(runs)}건: {[r['stamp'] for r in runs]}")

RANDOM_BASELINE = {
    'kobest_boolq': .5, 'kobest_copa': .5, 'kobest_wic': .5, 'kobest_sentineg': .5,
    'kobest_hellaswag': .25, 'kobest': None,
    'haerae': .2, 'haerae_general_knowledge': .2, 'haerae_history': .2,
    'haerae_loan_word': .2, 'haerae_rare_word': .2, 'haerae_standard_nomenclature': .2,
    'kmmlu': .25,
}

In [ ]:
# ---- 1) 최신 평가: 태스크별 정확도 vs 랜덤 기준선 ----
latest = runs[-1]
tasks = sorted(latest['accs'])
vals = [latest['accs'][t] for t in tasks]
base = [RANDOM_BASELINE.get(t) for t in tasks]

fig, ax = plt.subplots(figsize=(9, 0.45 * len(tasks) + 1.5))
bars = ax.barh(tasks, vals, color='#4c72b0', label='model acc')
for i, b in enumerate(base):
    if b is not None:
        ax.plot([b, b], [i - 0.4, i + 0.4], color='#c44e52', lw=2,
                label='random baseline' if i == 0 else None)
for i, v in enumerate(vals):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlim(0, 1)
ax.set_xlabel('accuracy')
ax.set_title(f"Benchmark accuracy vs random baseline  (run {latest['stamp']})")
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ---- 2) 평가 이력 추이: PPL + 주요 태스크 정확도 ----
stamps = [r['stamp'] for r in runs]
x = range(len(runs))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# PPL (로그 스케일 — 초기값이 수만 대라서)
ppls = [r['ppl'] for r in runs]
ax1.plot(x, ppls, marker='o', color='#4c72b0')
for i, p in enumerate(ppls):
    if p is not None:
        ax1.annotate(f'{p:,.0f}', (i, p), textcoords='offset points', xytext=(0, 8), fontsize=9)
ax1.set_yscale('log')
ax1.set_xticks(list(x), stamps, rotation=30, fontsize=8)
ax1.set_title('Held-out PPL (ko wiki) — lower is better')
ax1.grid(alpha=0.3)

# 정확도 추이 (모든 run에 공통으로 존재하는 태스크만)
common = set(runs[0]['accs'])
for r in runs[1:]:
    common &= set(r['accs'])
for t in sorted(common):
    ax2.plot(x, [r['accs'][t] for r in runs], marker='o', label=t, alpha=0.8)
ax2.set_xticks(list(x), stamps, rotation=30, fontsize=8)
ax2.set_ylim(0, 1)
ax2.set_title('Benchmark accuracy over runs — higher is better')
ax2.legend(fontsize=7, ncol=2)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

if len(runs) == 1:
    print("기록이 1건이라 추이는 점 하나입니다 — continue 모드로 추가 학습 후 다시 평가하면 곡선이 그려집니다.")